In [2]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.utils import check_random_state

from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest, ExtraSurvivalTrees
from sksurv.svm import FastKernelSurvivalSVM
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sklearn.base import clone

In [3]:
X_train = pd.read_excel("Data/X_train.xlsx").iloc[:, 1:]
X_test  = pd.read_excel("Data/X_test.xlsx").iloc[:, 1:]

y_train_df = pd.read_excel("Data/y_train.xlsx").iloc[:, 1:]
y_test_df  = pd.read_excel("Data/y_test.xlsx").iloc[:, 1:]

# Rename columns to "event" and "time"
y_train_df = y_train_df.rename(columns={"STATUS PFS": "event", "PFS": "time"})
y_test_df = y_test_df.rename(columns={"STATUS PFS": "event", "PFS": "time"})

y_train = Surv.from_dataframe("event", "time", y_train_df)
y_test  = Surv.from_dataframe("event", "time", y_test_df)

In [4]:
def cindex(y_true, y_pred):
    return concordance_index_censored(
        y_true["event"], y_true["time"], y_pred
    )[0]

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=31)

def stratify_labels(y):
    return y["event"].astype(int)

In [6]:
class SurvivalUnivariateSelector(BaseEstimator, TransformerMixin):
    def __init__(self, model, threshold=0.55, cv=3):
        self.model = model
        self.threshold = threshold
        self.cv = cv

    def fit(self, X, y):
        Xc = X.copy()
        self.selected_features_ = []

        skf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)
        strat_labels = stratify_labels(y)

        for col in Xc.columns:
            scores = []

            for train_idx, val_idx in skf.split(Xc, strat_labels):
                X_tr, X_val = Xc.iloc[train_idx][[col]], Xc.iloc[val_idx][[col]]
                y_tr, y_val = y[train_idx], y[val_idx]

                model = clone(self.model)
                model.fit(X_tr, y_tr)
                preds = model.predict(X_val)

                scores.append(cindex(y_val, preds))

            if np.mean(scores) > self.threshold:
                self.selected_features_.append(col)

        if len(self.selected_features_) == 0:
            self.selected_features_ = list(Xc.columns)

        return self

    def transform(self, X):
        Xc = X.copy()
        return Xc[self.selected_features_]

In [7]:
models = {
    "CPH": CoxPHSurvivalAnalysis(),
    "GBS": GradientBoostingSurvivalAnalysis(),
    "RSF": RandomSurvivalForest(),
    "EST": ExtraSurvivalTrees(),
    "SSVM": FastKernelSurvivalSVM()
}

param_grids = {
    "CPH": {"model__n_iter": [5,10,20,50,100,300], "model__alpha": [0.1,0.5,0.7,0.9], "model__ties": ['efron','breslow']},
    "GBS": {"model__n_estimators": [200, 300], "model__learning_rate": [0.02, 0.05], "model__max_depth": [2], "model__min_samples_leaf": [20, 30]},
    "RSF": {"model__max_depth": [2, 3, 4], "model__min_samples_leaf": [10, 20, 30], "model__min_samples_split": [10, 20, 30], "model__n_estimators": [100, 200]},
    "EST": {"model__n_estimators": [200, 400], "model__max_depth": [2, 3, 4], "model__min_samples_split": [10, 20, 30], "model__min_samples_leaf": [10, 20, 30]},
    "SSVM": {"model__kernel": ["rbf"], "model__alpha": [1e-3, 5e-3, 1e-2], "model__gamma": [1e-3, 5e-3, 1e-2], "model__tol": [1e-4], "model__max_iter": [2000]}
}

In [8]:
os.makedirs("Results", exist_ok=True)

results_path = "Results/survival_results.xlsx"

if os.path.exists(results_path):
    results_df = pd.read_excel(results_path)
else:
    results_df = pd.DataFrame()

done = set(zip(results_df.get("model", []), results_df.get("fs", [])))

results = []

for model_name, model in models.items():
    grid = param_grids[model_name]
    print(f"\nModel: {model_name}\n")

    for fs_flag in ["none", "univ"]:
        if (model_name, fs_flag) in done:
            print("  Skipping")
            continue

        print(f"  FS: {fs_flag}")

        try:
            # Only for FastKernelSurvivalSVM, replace 0 with 1e-7 in 'time'
            if model_name == "SSVM":
                y_train_df_mod = y_train_df.copy()
                y_test_df_mod = y_test_df.copy()
                y_train_df_mod["time"] = y_train_df_mod["time"].replace(0, 1e-7)
                y_test_df_mod["time"] = y_test_df_mod["time"].replace(0, 1e-7)
                y_train_mod = Surv.from_dataframe("event", "time", y_train_df_mod)
                y_test_mod = Surv.from_dataframe("event", "time", y_test_df_mod)
                X_train_mod = X_train.loc[y_train_df_mod.index]
                X_test_mod = X_test.loc[y_test_df_mod.index]
            else:
                y_train_mod = y_train
                y_test_mod = y_test
                X_train_mod = X_train
                X_test_mod = X_test

            if fs_flag == "univ":
                fs = SurvivalUnivariateSelector(model=model)
            else:
                fs = "passthrough"

            pipe = Pipeline([
                ("fs", fs),
                ("model", model)
            ])

            search = RandomizedSearchCV(
                pipe,
                param_distributions=grid,
                n_iter=20,
                cv=cv.split(X_train_mod, stratify_labels(y_train_mod)),
                scoring=lambda est, X, y: cindex(y, est.predict(X)),
                n_jobs=-1,
                random_state=25
            )

            search.fit(X_train_mod, y_train_mod)

            # Best estimator
            best_model = search.best_estimator_

            n_feats = best_model.named_steps["model"].n_features_in_
            if fs_flag == "univ":
                print(f"    Number of selected features: {n_feats}")

            hyperp = str(search.best_params_)
            print(f"    Hyperparameters of the best estimator:\n    {hyperp}")

            cv_c_mean = search.best_score_
            cv_c_std = search.cv_results_['std_test_score'][search.best_index_]

            # Evaluation
            train_c = cindex(y_train_mod, best_model.predict(X_train_mod))
            test_c  = cindex(y_test_mod, best_model.predict(X_test_mod))

            print("    C-index:")
            print(f"     - cv: {cv_c_mean} ({cv_c_std})")
            print(f"     - train: {train_c}")
            print(f"     - test: {test_c}")

            results.append({
                "model": model_name,
                "fs": fs_flag,
                "n features": n_feats,
                "hyperparameters": hyperp,
                "cv_cindex_mean": cv_c_mean,
                "cv_cindex_std": cv_c_std,
                "train_cindex": train_c,
                "test_cindex": test_c
            })

            # Save search and model
            #joblib.dump(search, f"Results/search_{model_name}_{fs_flag}.joblib")
            joblib.dump(best_model, f"Results/{model_name}_{fs_flag}.joblib")

        except Exception as e:
            print("Error:", e)
            results.append({
                "model": model_name,
                "fs": fs_flag,
                "error": str(e)
            })

        print("\n")
        # Save incrementally
        pd.DataFrame(results).to_excel(results_path, index=False)


Model: CPH

  FS: none
    Hyperparameters of the best estimator:
    {'model__ties': 'breslow', 'model__n_iter': 50, 'model__alpha': 0.9}
    C-index:
     - cv: 0.5811828505647443 (0.05588156946654852)
     - train: 0.6355480643758156
     - test: 0.6363934426229508


  FS: univ
    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__ties': 'breslow', 'model__n_iter': 50, 'model__alpha': 0.9}
    C-index:
     - cv: 0.5916397504917457 (0.04985986064799635)
     - train: 0.6280176163549369
     - test: 0.6516393442622951



Model: GBS

  FS: none


C:\Users\sferr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 8 is smaller than n_iter=20. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    Hyperparameters of the best estimator:
    {'model__n_estimators': 300, 'model__min_samples_leaf': 20, 'model__max_depth': 2, 'model__learning_rate': 0.02}
    C-index:
     - cv: 0.5893215357943339 (0.03834701143096083)
     - train: 0.7163440626359286
     - test: 0.659672131147541


  FS: univ


C:\Users\sferr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 8 is smaller than n_iter=20. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    Number of selected features: 7
    Hyperparameters of the best estimator:
    {'model__n_estimators': 200, 'model__min_samples_leaf': 20, 'model__max_depth': 2, 'model__learning_rate': 0.02}
    C-index:
     - cv: 0.5853542052787762 (0.028085721737153343)
     - train: 0.6796297303175294
     - test: 0.6262295081967213



Model: RSF

  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 10, 'model__max_depth': 3}
    C-index:
     - cv: 0.6121747392130802 (0.04169184644232843)
     - train: 0.6872553284036538
     - test: 0.6472131147540984


  FS: univ
    Number of selected features: 6
    Hyperparameters of the best estimator:
    {'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 10, 'model__max_depth': 3}
    C-index:
     - cv: 0.5841504487114804 (0.025371332944119206)
     - train: 0.6668592322749022
     - test: 0.6478688524590164



Model: EST



C:\Users\sferr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


    Hyperparameters of the best estimator:
    {'model__tol': 0.0001, 'model__max_iter': 2000, 'model__kernel': 'rbf', 'model__gamma': 0.01, 'model__alpha': 0.001}
    C-index:
     - cv: 0.6018210205191726 (0.054584926830299654)
     - train: 0.6426163549369291
     - test: 0.650655737704918


  FS: univ


C:\Users\sferr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\sferr\AppData\Local\Temp\ipykernel_43448\285613425.py:22: ConvergenceWarning: Optimization did not converge: Warning: Desired error not necessarily achieved due to precision loss.
  model.fit(X_tr, y_tr)
C:\Users\sferr\AppData\Local\Temp\ipykernel_43448\285613425.py:22: ConvergenceWarning: Optimization did not converge: Warning: Maximum number of iterations has been exceeded.
  model.fit(X_tr, y_tr)
C:\Users\sferr\AppData\Local\Temp\ipykernel_43448\285613425.py:22: ConvergenceWarning: Optimization did not converge: Warning: Desired error not necessarily achieved due to precision loss.
  model.fit(X_tr, y_tr)
C:\Users\sferr\AppData\Local\Temp\ipykerne

    Number of selected features: 7
    Hyperparameters of the best estimator:
    {'model__tol': 0.0001, 'model__max_iter': 2000, 'model__kernel': 'rbf', 'model__gamma': 0.001, 'model__alpha': 0.005}
    C-index:
     - cv: 0.612063502770064 (0.05228596599919195)
     - train: 0.6251087429317095
     - test: 0.6636065573770492


